In [1]:
# !pip install RapidFuzz

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import copy
from rapidfuzz import process, fuzz
import warnings

warnings.filterwarnings('ignore')

In [3]:

def check_all_code_columns(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df_columns_lower if "code" in column]

    return matching_columns


def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

def clean_string(s):
    return s.replace(':', '').replace('-', '').replace(')', '').replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()


In [4]:
df=pd.read_csv('vtaca2024rail_o2o_export_odbc.csv')
detail_df=pd.read_csv('O2O_STOPS_VTA.csv')
seq_df=pd.read_excel('details_vta_CA_od_excel (1).xlsx',sheet_name='STOPS')

In [5]:
code_columns=check_all_code_columns(df,df.columns)

code_columns=check_all_characters_present(df,code_columns)
columns_to_remove=[]
for column in code_columns:
    cleaned_column=clean_string(column)
#     if 'routesurveyed' in cleaned_column or 'language' in cleaned_column:
    if 'time' in cleaned_column or 'language' in cleaned_column:
        columns_to_remove.append(column)
code_columns

['ROUTE_SURVEYED_Code_',
 'BLUE_LINE_NB_ON_Code_',
 'BLUE_LINE_NB_OFF_Code_',
 'BLUE_LIME_SB_ON_Code_',
 'BLUE_LIME_SB_OFF_Code_',
 'GREEN_LINE_NB_ON_Code_',
 'GREEN_LINE_NB_OFF_Code_',
 'GREEN_LINE_SB_ON_Code_',
 'GREEN_LINE_SB_OFF_Code_',
 'ORANGE_LINE_EB_ON_Code_',
 'ORANGE_LINE_EB_OFF_Code_',
 'ORANGE_LINE_WB_ON_Code_',
 'ORANGE_LINE_WB_OFF_Code_',
 'TIME_BOARDED_Code_']

In [6]:
columns_to_remove

['TIME_BOARDED_Code_']

In [7]:
o2o_df=copy.deepcopy(df)

new_df=o2o_df.drop(columns=columns_to_remove)
new_df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,_INTERV_INIT,TIME_ADJUST,...,ORANGE_LINE_EB_ON_Code_,ORANGE_LINE_EB_ON,ORANGE_LINE_EB_OFF_Code_,ORANGE_LINE_EB_OFF,ORANGE_LINE_WB_ON_Code_,ORANGE_LINE_WB_ON,ORANGE_LINE_WB_OFF_Code_,ORANGE_LINE_WB_OFF,TIME_BOARDED,RESTART_URL
0,48568,23/09/2024 10:02,3,en,23/09/2024 10:02,23/09/2024 10:02,136.37.108.111,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,12:00 am - 4:59 am,NaN
1,48577,25/09/2024 07:39,3,en,25/09/2024 07:39,25/09/2024 07:39,12.15.243.210,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6:00 am - 9:59 am,NaN
2,48580,25/09/2024 11:25,3,en,25/09/2024 11:24,25/09/2024 11:25,172.56.47.61,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN
3,48587,25/09/2024 14:27,3,en,25/09/2024 14:09,25/09/2024 14:27,172.59.161.59,https://vta-ca-od.etc-research.com/,NaN,-25200,...,13.0,Great America,16.0,Baypointe,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN
4,48588,25/09/2024 14:29,3,en,25/09/2024 14:27,25/09/2024 14:29,172.59.161.59,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,17.0,Cisco Way,26.0,Alum Rock,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN


In [8]:
code_columns_check=check_all_characters_present(new_df,code_columns)

new_code_columns=check_all_characters_present(new_df,code_columns_check)


route_surveyed_column_check=['routesurveyed']
route_surveyed_column=check_all_characters_present(new_df,route_surveyed_column_check)
route_surveyed_column

['ROUTE_SURVEYED']

In [9]:
def get_line_name(row):
    if pd.notna(row['BLUE_LINE_NB_ON_Code_']) or pd.notna(row['BLUE_LINE_NB_OFF_Code_']):
        return 'BlueLine', row['BLUE_LINE_NB_ON']
    elif pd.notna(row['BLUE_LIME_SB_ON_Code_']) or pd.notna(row['BLUE_LIME_SB_OFF_Code_']):
        return 'BlueLine', row['BLUE_LIME_SB_ON']
    elif pd.notna(row['GREEN_LINE_NB_ON_Code_']) or pd.notna(row['GREEN_LINE_NB_OFF_Code_']):
        return 'GreenLine', row['GREEN_LINE_NB_ON']
    elif pd.notna(row['GREEN_LINE_SB_ON_Code_']) or pd.notna(row['GREEN_LINE_SB_OFF_Code_']):
        return 'GreenLine', row['GREEN_LINE_SB_ON']
    elif pd.notna(row['ORANGE_LINE_EB_ON_Code_']) or pd.notna(row['ORANGE_LINE_EB_OFF_Code_']):
        return 'OrangeLine', row['ORANGE_LINE_EB_ON']
    elif pd.notna(row['ORANGE_LINE_WB_ON_Code_']) or pd.notna(row['ORANGE_LINE_WB_OFF_Code_']):
        return 'OrangeLine', row['ORANGE_LINE_WB_ON']
    else:
        return 'Unknown', ''


In [10]:
# df['LineNameDatabase'] = df[code_columns[1:-1]].apply(get_line_name, axis=1)

# Apply the function to create the new columns
new_df[['Line_Value', 'STOP_NAME']] = df.apply(get_line_name, axis=1, result_type='expand')

In [11]:
new_df[['Line_Value']].head()

,Line_Value
0,GreenLine
1,BlueLine
2,BlueLine
3,OrangeLine
4,OrangeLine


In [12]:
def get_detail_route_line(x):
    values=str(x).split('_')
    if len(values)>1:
        return values[-2][0]
    else:
        return x
    
detail_df['Line_Value']=detail_df['ETC_ROUTE_ID'].apply(lambda x: str(x).split("_")[-2])

In [13]:
detail_df[['Line_Value']].head()

,Line_Value
0,GreenLine
1,GreenLine
2,GreenLine
3,GreenLine
4,GreenLine


In [14]:
code_columns

['ROUTE_SURVEYED_Code_',
 'BLUE_LINE_NB_ON_Code_',
 'BLUE_LINE_NB_OFF_Code_',
 'BLUE_LIME_SB_ON_Code_',
 'BLUE_LIME_SB_OFF_Code_',
 'GREEN_LINE_NB_ON_Code_',
 'GREEN_LINE_NB_OFF_Code_',
 'GREEN_LINE_SB_ON_Code_',
 'GREEN_LINE_SB_OFF_Code_',
 'ORANGE_LINE_EB_ON_Code_',
 'ORANGE_LINE_EB_OFF_Code_',
 'ORANGE_LINE_WB_ON_Code_',
 'ORANGE_LINE_WB_OFF_Code_',
 'TIME_BOARDED_Code_']

In [15]:
new_code_columns

['ROUTE_SURVEYED_Code_',
 'BLUE_LINE_NB_ON_Code_',
 'BLUE_LINE_NB_OFF_Code_',
 'BLUE_LIME_SB_ON_Code_',
 'BLUE_LIME_SB_OFF_Code_',
 'GREEN_LINE_NB_ON_Code_',
 'GREEN_LINE_NB_OFF_Code_',
 'GREEN_LINE_SB_ON_Code_',
 'GREEN_LINE_SB_OFF_Code_',
 'ORANGE_LINE_EB_ON_Code_',
 'ORANGE_LINE_EB_OFF_Code_',
 'ORANGE_LINE_WB_ON_Code_',
 'ORANGE_LINE_WB_OFF_Code_']

In [16]:
a = 'BLUE_LINE_NB_ON_Code_'
# Remove "Code" if it is present and join the rest
result = ''.join(part for part in a.split('_') if part.lower() != 'code').lower()

result


'bluelinenbon'

In [17]:
detail_df.columns

Index(['STOP_NAME', 'STOP_ID', 'ETC_STOP_NAME', 'note', 'ETC_STOP_ID',
       'ETC_ROUTE_ID', 'ETC_ROUTE_NAME', 'Line_Value'],
      dtype='object')

In [18]:

# for _,row in new_df.iterrows():
#     counter=0
#     for i in range(1,len(new_code_columns)):
#         seq_value=row[code_columns[i]]
#         print(f"{seq_value=}")
#         value_column_check=[''.join(part for part in code_columns[i].split('_') if part.lower() != 'code').lower()]
#         print(value_column_check)
#         value_column=check_all_characters_present(new_df,value_column_check)
#         print(f'{value_column=}')
#         value=row[value_column[0]]
#         print(value)
#         if not pd.isnull(value) and not pd.isnull(seq_value):
#             stop_value=value
#             print(f'{stop_value=}')
#             matched_value=detail_df[(detail_df['Line_Value']==row['Line_Value'])& (detail_df['ETC_STOP_NAME'].str.lower().str.contains(value.lower()))&(detail_df['STOP_ID']==seq_value)]
#             print(f'{matched_value=}')
#             if not matched_value.empty:
#                 counter+=1
#                 if counter==1:
#                     new_df.loc[row.name, ['ROUTE_DESCRIPTION[CODE]','ETC_ROUTE_NAME','BOARD_ID', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME']] = [
#                         matched_value.iloc[0]['ETC_ROUTE_ID'], 
#                         matched_value.iloc[0]['ETC_ROUTE_NAME'], 
#                         matched_value.iloc[0]['STOP_ID'], 
#                         matched_value.iloc[0]['ETC_STOP_ID'], 
#                         matched_value.iloc[0]['ETC_STOP_NAME'], 
#                         ]
#                 if counter>1:
#                     new_df.loc[row.name, ['ALIGHT_ID', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']] = [matched_value.iloc[0]['STOP_ID'], 
#                         matched_value.iloc[0]['ETC_STOP_ID'], 
#                         matched_value.iloc[0]['ETC_STOP_NAME']]


In [19]:
new_df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,_INTERV_INIT,TIME_ADJUST,...,ORANGE_LINE_EB_OFF_Code_,ORANGE_LINE_EB_OFF,ORANGE_LINE_WB_ON_Code_,ORANGE_LINE_WB_ON,ORANGE_LINE_WB_OFF_Code_,ORANGE_LINE_WB_OFF,TIME_BOARDED,RESTART_URL,Line_Value,STOP_NAME
0,48568,23/09/2024 10:02,3,en,23/09/2024 10:02,23/09/2024 10:02,136.37.108.111,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,12:00 am - 4:59 am,NaN,GreenLine,Winchester
1,48577,25/09/2024 07:39,3,en,25/09/2024 07:39,25/09/2024 07:39,12.15.243.210,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,6:00 am - 9:59 am,NaN,BlueLine,Santa Teresa
2,48580,25/09/2024 11:25,3,en,25/09/2024 11:24,25/09/2024 11:25,172.56.47.61,https://vta-ca-od.etc-research.com/,NaN,-25200,...,NaN,NaN,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,BlueLine,Orchard
3,48587,25/09/2024 14:27,3,en,25/09/2024 14:09,25/09/2024 14:27,172.59.161.59,https://vta-ca-od.etc-research.com/,NaN,-25200,...,16.0,Baypointe,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Great America
4,48588,25/09/2024 14:29,3,en,25/09/2024 14:27,25/09/2024 14:29,172.59.161.59,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,26.0,Alum Rock,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Cisco Way


In [20]:
ids_to_filter = [48588, 48589, 48590, 48591, 53004, 50571]

# Filtering new_df to get rows where 'id' is in ids_to_filter
new_df = new_df[new_df['id'].isin(ids_to_filter)]
new_df

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,_INTERV_INIT,TIME_ADJUST,...,ORANGE_LINE_EB_OFF_Code_,ORANGE_LINE_EB_OFF,ORANGE_LINE_WB_ON_Code_,ORANGE_LINE_WB_ON,ORANGE_LINE_WB_OFF_Code_,ORANGE_LINE_WB_OFF,TIME_BOARDED,RESTART_URL,Line_Value,STOP_NAME
4,48588,25/09/2024 14:29,3,en,25/09/2024 14:27,25/09/2024 14:29,172.59.161.59,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,26.0,Alum Rock,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Cisco Way
5,48589,25/09/2024 14:30,3,en,25/09/2024 14:29,25/09/2024 14:30,172.59.161.59,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,25.0,McKee,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Great America
6,48590,25/09/2024 14:32,3,en,25/09/2024 14:30,25/09/2024 14:32,172.59.161.59,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,19.0,Great Mall,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Champion
7,48591,25/09/2024 14:41,3,en,25/09/2024 14:32,25/09/2024 14:41,172.59.160.5,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,20.0,Milipitas,NaN,NaN,NaN,NaN,10:00 am - 2:59 pm,NaN,OrangeLine,Alder
1939,50571,30/09/2024 08:10,3,en,30/09/2024 08:03,30/09/2024 08:10,172.59.161.189,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,NaN,NaN,8.0,Great Mall,4.0,Berryessa,6:00 am - 9:59 am,NaN,OrangeLine,Great Mall
4208,53004,03/10/2024 16:37,3,en,03/10/2024 16:37,03/10/2024 16:37,107.77.213.108,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,NaN,NaN,21.0,Lockheed Martin,11.0,Baypointe,3:00 pm - 6:59 pm,NaN,OrangeLine,Lockheed Martin


In [21]:
for _, row in new_df.iterrows():
    counter = 0
    for i in range(1, len(new_code_columns)):
        seq_value = row[code_columns[i]]

        value_column_check = [''.join(part for part in code_columns[i].split('_') if part.lower() != 'code').lower()]
        print(value_column_check)
        value_column = check_all_characters_present(new_df, value_column_check)

        value = row[value_column[0]]

        if not pd.isnull(value) and not pd.isnull(seq_value):
            stop_value = value

            sequence_value = seq_df[(seq_df['route_short_name'] == row['Line_Value']) & (seq_df['ETC_STOP_NAME'].str.lower().str.contains(value.lower()))]
        
            seq_values = sequence_value[['seq_fixed']].values
            seq_values_array = np.array(seq_values).flatten()

            # Check if seq_values_array is not empty before proceeding
            if seq_values_array.size > 0:
                differences = np.abs(seq_values_array - seq_value)
                # Get the index of the minimum difference
                nearest_index = np.argmin(differences)

                # Retrieve the nearest value from seq_values_array using the index
                nearest_value = seq_values_array[nearest_index]
                matched_value = sequence_value[sequence_value['seq_fixed'] == nearest_value]

                if not matched_value.empty:
                    counter += 1
                    if counter == 1:
                        new_df.loc[row.name, ['ROUTE_DESCRIPTION[CODE]', 'ETC_ROUTE_NAME', 'BOARD_ID', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME']] = [
                            matched_value.iloc[0]['ETC_ROUTE_ID'],
                            matched_value.iloc[0]['ETC_ROUTE_NAME'],
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
                    if counter > 1:
                        new_df.loc[row.name, ['ALIGHT_ID', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']] = [
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
            else:
                print(f"No valid seq_fixed values found for {row['Line_Value']} and {value}.")


['bluelinenbon']
['bluelinenboff']
['bluelimesbon']
['bluelimesboff']
['greenlinenbon']
['greenlinenboff']
['greenlinesbon']
['greenlinesboff']
['orangelineebon']
No valid seq_fixed values found for OrangeLine and Cisco Way.
['orangelineeboff']
['orangelinewbon']
['orangelinewboff']
['bluelinenbon']
['bluelinenboff']
['bluelimesbon']
['bluelimesboff']
['greenlinenbon']
['greenlinenboff']
['greenlinesbon']
['greenlinesboff']
['orangelineebon']
['orangelineeboff']
['orangelinewbon']
['orangelinewboff']
['bluelinenbon']
['bluelinenboff']
['bluelimesbon']
['bluelimesboff']
['greenlinenbon']
['greenlinenboff']
['greenlinesbon']
['greenlinesboff']
['orangelineebon']
['orangelineeboff']
['orangelinewbon']
['orangelinewboff']
['bluelinenbon']
['bluelinenboff']
['bluelimesbon']
['bluelimesboff']
['greenlinenbon']
['greenlinenboff']
['greenlinesbon']
['greenlinesboff']
['orangelineebon']
['orangelineeboff']
No valid seq_fixed values found for OrangeLine and Milipitas.
['orangelinewbon']
['orange

In [22]:
from fuzzywuzzy import fuzz, process  # Or use from rapidfuzz import fuzz, process for faster matching

for _, row in new_df.iterrows():
    counter = 0
    for i in range(1, len(new_code_columns)):
        seq_value = row[code_columns[i]]
        value_column_check = [''.join(part for part in code_columns[i].split('_') if part.lower() != 'code').lower()]
        value_column = check_all_characters_present(new_df, value_column_check)
        value = row[value_column[0]]

        if pd.isnull(value) or pd.isnull(seq_value):
            continue  # Skip if value or seq_value is null

        stop_value = value

        # First, filter sequence_value using the original logic
        sequence_value = seq_df[
            (seq_df['route_short_name'] == row['Line_Value']) & 
            (seq_df['ETC_STOP_NAME'].str.lower().str.contains(value.lower()))
        ]

        # If sequence_value is empty, apply fuzzy matching
        if sequence_value.empty:
            # Use fuzzy matching to find the best match for ETC_STOP_NAME
            best_match, score, _ = process.extractOne(
                value.lower(),
                seq_df[seq_df['route_short_name'] == row['Line_Value']]['ETC_STOP_NAME'].str.lower(),
                scorer=fuzz.ratio
            )
            print(score,best_match)
            print()
            if score >= 40:
                sequence_value = seq_df[
                    (seq_df['route_short_name'] == row['Line_Value']) & 
                    (seq_df['ETC_STOP_NAME'].str.lower() == best_match)
                ]
                
        # Proceed only if we have a valid sequence_value after both methods
        if not sequence_value.empty:
            seq_values_array = sequence_value['seq_fixed'].values

            if seq_values_array.size > 0:
                differences = np.abs(seq_values_array - seq_value)
                nearest_index = np.argmin(differences)
                nearest_value = seq_values_array[nearest_index]
                matched_value = sequence_value[sequence_value['seq_fixed'] == nearest_value]

                if not matched_value.empty:
                    counter += 1
                    if counter == 1:
                        new_df.loc[row.name, ['ROUTE_DESCRIPTION[CODE]', 'ETC_ROUTE_NAME', 'BOARD_ID', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME']] = [
                            matched_value.iloc[0]['ETC_ROUTE_ID'],
                            matched_value.iloc[0]['ETC_ROUTE_NAME'],
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
                    else:  # counter > 1
                        new_df.loc[row.name, ['ALIGHT_ID', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']] = [
                            matched_value.iloc[0]['seq_fixed'],
                            matched_value.iloc[0]['ETC_STOP_ID'],
                            matched_value.iloc[0]['ETC_STOP_NAME'],
                        ]
            else:
                print(f"No valid seq_fixed values found for {row['Line_Value']} and {value}.")


64 cisco station

64 milpitas station



In [23]:
'VTA_2_OrangeLine_01_4805'.split('_')[-2] not in 'VTA_2_OrangeLine_00_4805'.split('_')

True

In [24]:
new_df[['id','ROUTE_DESCRIPTION[CODE]', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']]

,id,ROUTE_DESCRIPTION[CODE],BOARD_STOP[CODE],BOARD_STOP_NAME,ALIGHT_STOP[CODE],ALIGHT_STOP_NAME
4,48588,VTA_2_OrangeLine_00,VTA_2_OrangeLine_00_4802,Cisco Station,VTA_2_OrangeLine_00_5242,Alum Rock Station
5,48589,VTA_2_OrangeLine_00,VTA_2_OrangeLine_00_4798,Great America Station,VTA_2_OrangeLine_00_5241,Mckee Station
6,48590,VTA_2_OrangeLine_00,VTA_2_OrangeLine_00_4800,Champion Station,VTA_2_OrangeLine_00_5235,Great Mall Station
7,48591,VTA_2_OrangeLine_00,VTA_2_OrangeLine_00_4803,Alder Station,VTA_2_OrangeLine_00_5236,Milpitas Station
1939,50571,VTA_2_OrangeLine_01,VTA_2_OrangeLine_01_5250,Great Mall Station,VTA_2_OrangeLine_01_5246,Berryessa Station
4208,53004,VTA_2_OrangeLine_01,VTA_2_OrangeLine_01_4816,Lockheed Martin Station,VTA_2_OrangeLine_01_4806,Baypointe Station


In [25]:
new_df=pd.merge(new_df,df[['id','TIME_BOARDED_Code_']],on='id',how='left')

In [26]:
new_df['Date'] = np.where(new_df['Completed'].notna(), new_df['Completed'], new_df['Date_started'])

In [27]:
def get_day_name(x):
    if pd.isna(x):
        return np.nan
    x = str(x)
    try:
        date_object = datetime.strptime(x, '%Y-%m-%d %H:%M:%S')
    except ValueError:
        try:
            date_object = datetime.strptime(x, '%d/%m/%Y %H:%M')
        except ValueError:
            return np.nan
    day_name = date_object.strftime('%A')
    if day_name in ['Saturday', 'Sunday']:
        return 'Weekend'
    return 'Weekday'

new_df['Day']=new_df['Date'].apply(get_day_name)

In [28]:
'milipitas' in 'milipitas station'

True

In [29]:
na_columns_check=['routedescritioncode','boardid','boardstopcode','boardstopname','alightid','alightstopcode','alightstopname']
na_columns=check_all_characters_present(new_df,na_columns_check)
print(na_columns)
print(new_df.columns)



['BOARD_ID', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME', 'ALIGHT_ID', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']
Index(['id', 'Completed', 'Last_page', 'Start_language', 'Date_started',
       'Date_last_action', 'IP_address', 'Referring_URL', '_INTERV_INIT',
       'TIME_ADJUST', 'ORIGRESPID', 'ROUTE_SURVEYED_Code_', 'ROUTE_SURVEYED',
       'BLUE_LINE_NB_ON_Code_', 'BLUE_LINE_NB_ON', 'BLUE_LINE_NB_OFF_Code_',
       'BLUE_LINE_NB_OFF', 'BLUE_LIME_SB_ON_Code_', 'BLUE_LIME_SB_ON',
       'BLUE_LIME_SB_OFF_Code_', 'BLUE_LIME_SB_OFF', 'GREEN_LINE_NB_ON_Code_',
       'GREEN_LINE_NB_ON', 'GREEN_LINE_NB_OFF_Code_', 'GREEN_LINE_NB_OFF',
       'GREEN_LINE_SB_ON_Code_', 'GREEN_LINE_SB_ON', 'GREEN_LINE_SB_OFF_Code_',
       'GREEN_LINE_SB_OFF', 'ORANGE_LINE_EB_ON_Code_', 'ORANGE_LINE_EB_ON',
       'ORANGE_LINE_EB_OFF_Code_', 'ORANGE_LINE_EB_OFF',
       'ORANGE_LINE_WB_ON_Code_', 'ORANGE_LINE_WB_ON',
       'ORANGE_LINE_WB_OFF_Code_', 'ORANGE_LINE_WB_OFF', 'TIME_BOARDED',
       'RESTART_URL', 'Lin

In [30]:
new_df.dropna(subset=['ROUTE_DESCRIPTION[CODE]', 'BOARD_STOP[CODE]', 'BOARD_STOP_NAME', 'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME'],inplace=True)


In [31]:
time_period_mapping = {
    'AM1':'AM',
    'AM2':'AM',
    'AM3':'AM',
    'PM1': 'PM',
    'PM2': 'Evening',
    'MID': 'MIDDAY',
    'AM': 'AM'
}

time_period_code_mapping = {
    'PM':2,
    'Evening':4,
    'MIDDAY':1,
    'AM': 0
}

In [32]:
new_df['TIME PERIOD'] = new_df['TIME_BOARDED_Code_'].map(time_period_mapping).fillna(new_df['TIME_BOARDED_Code_'])
new_df['TIME PERIOD[Code]'] = new_df['TIME PERIOD'].map(time_period_code_mapping)

In [33]:
# # Perfectly working 
# for _, row in new_df.iterrows():
#     # Extract the alight sequence
#     alight_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
#                            (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])][['seq_fixed']]
    
#     print(row['ROUTE_DESCRIPTION[CODE]'])
#     print(row['ALIGHT_STOP[CODE]'])
#     if not alight_seq_df.empty:
#         alight_seq = alight_seq_df.iloc[0, 0]
#     else:
#         alight_seq = 0
    
#     # Extract the board sequence
#     board_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
#                           (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])][['seq_fixed']]
#     print(row['ROUTE_DESCRIPTION[CODE]'])
#     print(row['BOARD_STOP[CODE]'])
#     if not board_seq_df.empty:
#         board_seq = board_seq_df.iloc[0, 0]
#     else:
#         board_seq = 0

#     # Assign values to new_df
#     new_df.loc[row.name, 'Alight_Seq'] = alight_seq
#     new_df.loc[row.name, 'Board_Seq'] = board_seq

# new_df['SEQ_CHECK'] = new_df['Alight_Seq'] - new_df['Board_Seq']

In [34]:
# Changes to add Alighting and Boarding Lat Long
for _, row in new_df.iterrows():
    # Extract the alight sequence
    alight_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                           (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME']) ][['seq_fixed']]
    alight_seq_lat_long_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                                    (seq_df['ETC_STOP_ID'] == row['ALIGHT_STOP[CODE]'])][['stop_lat6', 'stop_lon6']]
    
    if not alight_seq_df.empty:
        alight_seq = alight_seq_df.iloc[0, 0]
    else:
        alight_seq = 0

    # Extract the board sequence
    board_seq_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                          (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])][['seq_fixed']]
    board_seq_lat_long_df = seq_df[(seq_df['ETC_ROUTE_NAME'] == row['ETC_ROUTE_NAME']) & 
                                   (seq_df['ETC_STOP_ID'] == row['BOARD_STOP[CODE]'])][['stop_lat6', 'stop_lon6']]
    
    if not board_seq_df.empty:
        board_seq = board_seq_df.iloc[0, 0]
    else:
        board_seq = 0

    # Assign sequence values to new_df
    new_df.loc[row.name, 'Alight_Seq'] = alight_seq
    new_df.loc[row.name, 'Board_Seq'] = board_seq

    # Assign latitude and longitude values to new_df
    if not board_seq_lat_long_df.empty:
        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = board_seq_lat_long_df.iloc[0, 0]
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = board_seq_lat_long_df.iloc[0, 1]
    else:
        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = None
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = None

    if not alight_seq_lat_long_df.empty:
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = alight_seq_lat_long_df.iloc[0, 0]
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = alight_seq_lat_long_df.iloc[0, 1]
    else:
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = None
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = None
        
new_df['SEQ_CHECK'] = new_df['Alight_Seq'] - new_df['Board_Seq']

In [35]:
new_df[new_df['SEQ_CHECK']<0]

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,_INTERV_INIT,TIME_ADJUST,...,Day,TIME PERIOD,TIME PERIOD[Code],Alight_Seq,Board_Seq,BOARD_STOP_ON_LAT,BOARD_STOP_ON_LONG,ALIGHT_STOP_ON_LAT,ALIGHT_STOP_ON_LONG,SEQ_CHECK
4,50571,30/09/2024 08:10,3,en,30/09/2024 08:03,30/09/2024 08:10,172.59.161.189,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,Weekday,AM,0,4.0,8.0,37.414390,-121.902123,37.389085,-121.863113,-4.0
5,53004,03/10/2024 16:37,3,en,03/10/2024 16:37,03/10/2024 16:37,107.77.213.108,https://vta-ca-od.etc-research.com/index.php/s...,NaN,-25200,...,Weekday,PM,2,11.0,21.0,37.409404,-122.027132,37.410531,-121.942317,-10.0


In [36]:
new_df.columns

Index(['id', 'Completed', 'Last_page', 'Start_language', 'Date_started',
       'Date_last_action', 'IP_address', 'Referring_URL', '_INTERV_INIT',
       'TIME_ADJUST', 'ORIGRESPID', 'ROUTE_SURVEYED_Code_', 'ROUTE_SURVEYED',
       'BLUE_LINE_NB_ON_Code_', 'BLUE_LINE_NB_ON', 'BLUE_LINE_NB_OFF_Code_',
       'BLUE_LINE_NB_OFF', 'BLUE_LIME_SB_ON_Code_', 'BLUE_LIME_SB_ON',
       'BLUE_LIME_SB_OFF_Code_', 'BLUE_LIME_SB_OFF', 'GREEN_LINE_NB_ON_Code_',
       'GREEN_LINE_NB_ON', 'GREEN_LINE_NB_OFF_Code_', 'GREEN_LINE_NB_OFF',
       'GREEN_LINE_SB_ON_Code_', 'GREEN_LINE_SB_ON', 'GREEN_LINE_SB_OFF_Code_',
       'GREEN_LINE_SB_OFF', 'ORANGE_LINE_EB_ON_Code_', 'ORANGE_LINE_EB_ON',
       'ORANGE_LINE_EB_OFF_Code_', 'ORANGE_LINE_EB_OFF',
       'ORANGE_LINE_WB_ON_Code_', 'ORANGE_LINE_WB_ON',
       'ORANGE_LINE_WB_OFF_Code_', 'ORANGE_LINE_WB_OFF', 'TIME_BOARDED',
       'RESTART_URL', 'Line_Value', 'STOP_NAME', 'ROUTE_DESCRIPTION[CODE]',
       'ETC_ROUTE_NAME', 'BOARD_ID', 'BOARD_STOP[CODE]',

In [37]:
new_df = new_df[new_df['SEQ_CHECK'] != 0].copy()

In [38]:
a='VTA_2_OrangeLine_00'
f"{'_'.join(a.split('_')[:-1])}_01" if a.split('_')[-1] == '00' else f"{'_'.join(a.split('_')[:-1])}_00"

'VTA_2_OrangeLine_01'

In [39]:
new_df[['id','ROUTE_DESCRIPTION[CODE]','Board_Seq' ,'BOARD_STOP[CODE]', 'BOARD_STOP_NAME', 'Alight_Seq','ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']]

,id,ROUTE_DESCRIPTION[CODE],Board_Seq,BOARD_STOP[CODE],BOARD_STOP_NAME,Alight_Seq,ALIGHT_STOP[CODE],ALIGHT_STOP_NAME
0,48588,VTA_2_OrangeLine_00,18.0,VTA_2_OrangeLine_00_4802,Cisco Station,27.0,VTA_2_OrangeLine_00_5242,Alum Rock Station
1,48589,VTA_2_OrangeLine_00,13.0,VTA_2_OrangeLine_00_4798,Great America Station,26.0,VTA_2_OrangeLine_00_5241,Mckee Station
2,48590,VTA_2_OrangeLine_00,15.0,VTA_2_OrangeLine_00_4800,Champion Station,20.0,VTA_2_OrangeLine_00_5235,Great Mall Station
3,48591,VTA_2_OrangeLine_00,19.0,VTA_2_OrangeLine_00_4803,Alder Station,21.0,VTA_2_OrangeLine_00_5236,Milpitas Station
4,50571,VTA_2_OrangeLine_01,8.0,VTA_2_OrangeLine_01_5250,Great Mall Station,4.0,VTA_2_OrangeLine_01_5246,Berryessa Station
5,53004,VTA_2_OrangeLine_01,21.0,VTA_2_OrangeLine_01_4816,Lockheed Martin Station,11.0,VTA_2_OrangeLine_01_4806,Baypointe Station


In [40]:
new_df[['id','ROUTE_DESCRIPTION[CODE]', 'Board_Seq','BOARD_STOP[CODE]', 'BOARD_STOP_NAME','Alight_Seq' ,'ALIGHT_STOP[CODE]', 'ALIGHT_STOP_NAME']]

,id,ROUTE_DESCRIPTION[CODE],Board_Seq,BOARD_STOP[CODE],BOARD_STOP_NAME,Alight_Seq,ALIGHT_STOP[CODE],ALIGHT_STOP_NAME
0,48588,VTA_2_OrangeLine_00,18.0,VTA_2_OrangeLine_00_4802,Cisco Station,27.0,VTA_2_OrangeLine_00_5242,Alum Rock Station
1,48589,VTA_2_OrangeLine_00,13.0,VTA_2_OrangeLine_00_4798,Great America Station,26.0,VTA_2_OrangeLine_00_5241,Mckee Station
2,48590,VTA_2_OrangeLine_00,15.0,VTA_2_OrangeLine_00_4800,Champion Station,20.0,VTA_2_OrangeLine_00_5235,Great Mall Station
3,48591,VTA_2_OrangeLine_00,19.0,VTA_2_OrangeLine_00_4803,Alder Station,21.0,VTA_2_OrangeLine_00_5236,Milpitas Station
4,50571,VTA_2_OrangeLine_01,8.0,VTA_2_OrangeLine_01_5250,Great Mall Station,4.0,VTA_2_OrangeLine_01_5246,Berryessa Station
5,53004,VTA_2_OrangeLine_01,21.0,VTA_2_OrangeLine_01_4816,Lockheed Martin Station,11.0,VTA_2_OrangeLine_01_4806,Baypointe Station


In [41]:
# for _, row in new_df.iterrows():
#     if row['SEQ_CHECK'] < 0:
#         route_code=row['ROUTE_DESCRIPTION[CODE]']
#         new_route_code=f"{'_'.join(route_code.split('_')[:-1])}_01" if route_code.split('_')[-1] == '00' else f"{'_'.join(route_code.split('_')[:-1])}_00"
#         new_route_name=seq_df[seq_df['ETC_ROUTE_ID']==new_route_code][['ETC_ROUTE_NAME']].iloc[0,0]


#         # Swap the board and alight value
#         # board_id = row['BOARD_ID']
#         # board_seq = row['Board_Seq']
#         # board_stop_code = row['BOARD_STOP[CODE]']
#         # board_stop_name = row['BOARD_STOP_NAME']
#         # board_stop_lat=row['BOARD_STOP_ON_LAT']
#         # board_stop_lon=row['BOARD_STOP_ON_LONG']
#         # alight_id = row['ALIGHT_ID']
#         # alight_seq = row['Alight_Seq']
#         # alight_stop_code = row['ALIGHT_STOP[CODE]']
#         # alight_stop_name = row['ALIGHT_STOP_NAME']
#         # alight_stop_lat=row['ALIGHT_STOP_ON_LAT']
#         # alight_stop_lon=row['ALIGHT_STOP_ON_LONG']
        
#         # new_df.loc[row.name, 'ROUTE_DESCRIPTION[CODE]'] = new_route_code
        
        
#         new_df.loc[row.name, 'ROUTE_DESCRIPTION[CODE]'] = new_route_code
#         new_df.loc[row.name,'ETC_ROUTE_NAME']=new_route_name
#         ids_list.append(row['id'])

#         # new_df.loc[row.name, 'BOARD_ID'] = alight_id
#         # new_df.loc[row.name, 'Board_Seq'] = alight_seq
#         # new_df.loc[row.name, 'BOARD_STOP[CODE]'] = alight_stop_code
#         # new_df.loc[row.name, 'BOARD_STOP_NAME'] = alight_stop_name
#         # new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = alight_stop_lat
#         # new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = alight_stop_lon
        
#         # new_df.loc[row.name, 'ALIGHT_ID'] = board_id
#         # new_df.loc[row.name, 'Alight_Seq'] = board_seq
#         # new_df.loc[row.name, 'ALIGHT_STOP[CODE]'] = board_stop_code
#         # new_df.loc[row.name, 'ALIGHT_STOP_NAME'] = board_stop_name
#         # new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = board_stop_lat
#         # new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = board_stop_lon

In [42]:
for _, row in new_df.iterrows():
    if row['SEQ_CHECK'] < 0:
        # Extract route code and modify it
        route_code = row['ROUTE_DESCRIPTION[CODE]']
        new_route_code = f"{'_'.join(route_code.split('_')[:-1])}_01" if route_code.split('_')[-1] == '00' else f"{'_'.join(route_code.split('_')[:-1])}_00"
        
        # Get new route name
        new_route_name = seq_df[seq_df['ETC_ROUTE_ID'] == new_route_code]['ETC_ROUTE_NAME'].iloc[0]

        # Board stop details
        board_stop_code = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])]['ETC_STOP_ID'].iloc[0]
        board_stop_lat_lon = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['BOARD_STOP_NAME'])][['seq_fixed', 'stop_lat6', 'stop_lon6']]

        # Alight stop details
        alight_stop_code = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])]['ETC_STOP_ID'].iloc[0]
        alight_stop_lat_lon = seq_df[(seq_df['ETC_ROUTE_ID'] == new_route_code) & (seq_df['ETC_STOP_NAME'] == row['ALIGHT_STOP_NAME'])][['seq_fixed', 'stop_lat6', 'stop_lon6']]

        # Update new_df with the retrieved values
        new_df.loc[row.name, 'ROUTE_DESCRIPTION[CODE]'] = new_route_code
        new_df.loc[row.name, 'ETC_ROUTE_NAME'] = new_route_name

        new_df.loc[row.name, 'Board_Seq'] = board_stop_lat_lon['seq_fixed'].iloc[0]
        new_df.loc[row.name, 'BOARD_STOP[CODE]'] = board_stop_code 

        new_df.loc[row.name, 'BOARD_STOP_ON_LAT'] = board_stop_lat_lon['stop_lat6'].iloc[0] 
        new_df.loc[row.name, 'BOARD_STOP_ON_LONG'] = board_stop_lat_lon['stop_lon6'].iloc[0] 

        new_df.loc[row.name, 'Alight_Seq'] = alight_stop_lat_lon['seq_fixed'].iloc[0]
        new_df.loc[row.name, 'ALIGHT_STOP[CODE]'] = alight_stop_code

        new_df.loc[row.name, 'ALIGHT_STOP_ON_LAT'] = alight_stop_lat_lon['stop_lat6'].iloc[0]
        new_df.loc[row.name, 'ALIGHT_STOP_ON_LONG'] = alight_stop_lat_lon['stop_lon6'].iloc[0] 


In [43]:
seq_df[seq_df['ETC_ROUTE_ID']=='VTA_2_BlueLine_01'][['ETC_ROUTE_NAME']].iloc[0,0]

'Blue Line - Baypointe - Santa Teresa [Southbound Light Rail Toward St Theresa]'

In [44]:
seq_df[seq_df['ETC_ROUTE_ID']=='VTA_2_BlueLine_00'][['ETC_ROUTE_NAME']].iloc[0:5,0]

3882    Blue Line - Baypointe - Santa Teresa [Northbou...
3883    Blue Line - Baypointe - Santa Teresa [Northbou...
3884    Blue Line - Baypointe - Santa Teresa [Northbou...
3885    Blue Line - Baypointe - Santa Teresa [Northbou...
3886    Blue Line - Baypointe - Santa Teresa [Northbou...
Name: ETC_ROUTE_NAME, dtype: object

In [45]:
# seq_df[(seq_df['ETC_ROUTE_ID']=='VTA_2_BlueLine_00')|(seq_df['ETC_ROUTE_ID']=='VTA_2_BlueLine_01')][['ETC_ROUTE_ID','ETC_ROUTE_NAME']].to_csv('ROUTE_VALUES.csv',index=False)

In [46]:

final_df=new_df[['id','Date','Day','TIME_BOARDED_Code_','TIME_BOARDED','TIME PERIOD','TIME PERIOD[Code]','ROUTE_DESCRIPTION[CODE]','ETC_ROUTE_NAME','Line_Value','BOARD_STOP[CODE]','Board_Seq' ,'BOARD_STOP_NAME','BOARD_STOP_ON_LAT','BOARD_STOP_ON_LONG','ALIGHT_STOP[CODE]','Alight_Seq' ,'ALIGHT_STOP_NAME','ALIGHT_STOP_ON_LAT','ALIGHT_STOP_ON_LONG']]
# .to_csv('VTA_O2O_002.csv',index=False)

In [47]:
final_df['TIME_BOARDED'].unique

<bound method Series.unique of 0    10:00 am - 2:59 pm
1    10:00 am - 2:59 pm
2    10:00 am - 2:59 pm
3    10:00 am - 2:59 pm
4     6:00 am - 9:59 am
5     3:00 pm - 6:59 pm
Name: TIME_BOARDED, dtype: object>

In [48]:
final_df.head()

,id,Date,Day,TIME_BOARDED_Code_,TIME_BOARDED,TIME PERIOD,TIME PERIOD[Code],ROUTE_DESCRIPTION[CODE],ETC_ROUTE_NAME,Line_Value,BOARD_STOP[CODE],Board_Seq,BOARD_STOP_NAME,BOARD_STOP_ON_LAT,BOARD_STOP_ON_LONG,ALIGHT_STOP[CODE],Alight_Seq,ALIGHT_STOP_NAME,ALIGHT_STOP_ON_LAT,ALIGHT_STOP_ON_LONG
0,48588,25/09/2024 14:29,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4802,18.0,Cisco Station,37.412348,-121.928312,VTA_2_OrangeLine_00_5242,27.0,Alum Rock Station,37.358134,-121.832120
1,48589,25/09/2024 14:30,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4798,13.0,Great America Station,37.403555,-121.974058,VTA_2_OrangeLine_00_5241,26.0,Mckee Station,37.371057,-121.844036
2,48590,25/09/2024 14:32,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4800,15.0,Champion Station,37.408898,-121.951936,VTA_2_OrangeLine_00_5235,20.0,Great Mall Station,37.413925,-121.901434
3,48591,25/09/2024 14:41,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4803,19.0,Alder Station,37.413572,-121.917118,VTA_2_OrangeLine_00_5236,21.0,Milpitas Station,37.408549,-121.890618
4,50571,30/09/2024 08:10,Weekday,AM3,6:00 am - 9:59 am,AM,0,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_5235,20.0,Great Mall Station,37.413925,-121.901434,VTA_2_OrangeLine_00_5239,24.0,Berryessa Station,37.387386,-121.861477


In [49]:
final_df.columns

Index(['id', 'Date', 'Day', 'TIME_BOARDED_Code_', 'TIME_BOARDED',
       'TIME PERIOD', 'TIME PERIOD[Code]', 'ROUTE_DESCRIPTION[CODE]',
       'ETC_ROUTE_NAME', 'Line_Value', 'BOARD_STOP[CODE]', 'Board_Seq',
       'BOARD_STOP_NAME', 'BOARD_STOP_ON_LAT', 'BOARD_STOP_ON_LONG',
       'ALIGHT_STOP[CODE]', 'Alight_Seq', 'ALIGHT_STOP_NAME',
       'ALIGHT_STOP_ON_LAT', 'ALIGHT_STOP_ON_LONG'],
      dtype='object')

In [50]:
final_df['TIME_BOARDED_Code_'].unique()

array(['MID', 'AM3', 'PM1'], dtype=object)

In [51]:
final_df[['TIME_BOARDED','TIME PERIOD[Code]','TIME PERIOD','TIME_BOARDED_Code_']]

,TIME_BOARDED,TIME PERIOD[Code],TIME PERIOD,TIME_BOARDED_Code_
0,10:00 am - 2:59 pm,1,MIDDAY,MID
1,10:00 am - 2:59 pm,1,MIDDAY,MID
2,10:00 am - 2:59 pm,1,MIDDAY,MID
3,10:00 am - 2:59 pm,1,MIDDAY,MID
4,6:00 am - 9:59 am,0,AM,AM3
5,3:00 pm - 6:59 pm,2,PM,PM1


In [52]:
final_df[['BOARD_STOP[CODE]']].iloc[0,0]

'VTA_2_OrangeLine_00_4802'

In [53]:
final_df[final_df['BOARD_STOP[CODE]']=='VTA_2_OrangeLine_00_4801']['Board_Seq']

Series([], Name: Board_Seq, dtype: float64)

In [54]:
stop_id='VTA_2_OrangeLine_00_4801'

In [55]:
seq_df[seq_df['ETC_STOP_ID']==stop_id]
# seq_df[seq_df['ETC_STOP_ID']==final_df[['BOARD_STOP[CODE]']].iloc[0,0]]

,agency,gtfs_ver,gtfs_date,route_short_name,route_long_name,direction_id,route_dir,seq_fixed,stop_id,stop_name,stop_lat,stop_lon,stop_lat6,stop_lon6,ETC_STOP_ID,ETC_STOP_NAME,notes,ETC_ROUTE_ID,ETC_ROUTE_NAME
4547,VTA,2,8/7/2024,OrangeLine,Mountain View - Alum Rock,0,OrangeLine_0,17,4801,Baypointe Station,37.410877,-121.941523,37.410877,-121.941523,VTA_2_OrangeLine_00_4801,Baypointe Station,NaN,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...


In [56]:
for _,row in final_df.iterrows():
    detail_board_seq=seq_df[seq_df['ETC_STOP_ID']==row['BOARD_STOP[CODE]']][['seq_fixed']].iloc[0,0]
    detail_alight_seq=seq_df[seq_df['ETC_STOP_ID']==row['ALIGHT_STOP[CODE]']][['seq_fixed']].iloc[0,0]
    if detail_board_seq!=row['Board_Seq']:
        final_df.loc[row.name,'Board_Seq']=detail_board_seq
    if detail_alight_seq!=row['Alight_Seq']:
        final_df.loc[row.name,'Alight_Seq']=detail_alight_seq

In [57]:
mydata_df=pd.read_excel('O2O_20241024_VTA_rail_006.xlsx',sheet_name='O2O Data')
mydata_df.head()

,id,Date,Day,TIME_BOARDED_Code_,TIME_BOARDED,TIME PERIOD,TIME PERIOD[Code],ROUTE_DESCRIPTION[CODE],ETC_ROUTE_NAME,Line_Value,BOARD_STOP[CODE],Board_Seq,BOARD_STOP_NAME,BOARD_STOP_ON_LAT,BOARD_STOP_ON_LONG,ALIGHT_STOP[CODE],Alight_Seq,ALIGHT_STOP_NAME,ALIGHT_STOP_ON_LAT,ALIGHT_STOP_ON_LONG
0,48577,25/09/2024 07:39,Weekday,AM3,6:00 am - 9:59 am,AM,0,VTA_2_BlueLine_00,Blue Line - Baypointe - Santa Teresa [Northbou...,BlueLine,VTA_2_BlueLine_00_4736,1,Santa Teresa Station,37.236668,-121.789141,VTA_2_BlueLine_00_4743,9,Tamien Station,37.311927,-121.884809
1,48580,25/09/2024 11:25,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_BlueLine_01,Blue Line - Baypointe - Santa Teresa [Southbou...,BlueLine,VTA_2_BlueLine_01_4764,4,Orchard Station,37.394082,-121.933837,VTA_2_BlueLine_01_4768,8,Metro/Airport Station,37.368664,-121.914883
2,48587,25/09/2024 14:27,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4798,13,Great America Station,37.403555,-121.974058,VTA_2_OrangeLine_00_4760,16,Baypointe Station,37.410779,-121.941526
3,48588,25/09/2024 14:29,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4802,18,Cisco Station,37.412348,-121.928312,VTA_2_OrangeLine_00_5242,27,Alum Rock Station,37.358134,-121.832120
4,48589,25/09/2024 14:30,Weekday,MID,10:00 am - 2:59 pm,MIDDAY,1,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,OrangeLine,VTA_2_OrangeLine_00_4798,13,Great America Station,37.403555,-121.974058,VTA_2_OrangeLine_00_5241,26,Mckee Station,37.371057,-121.844036


In [58]:
mydata_df['TIME_BOARDED'].unique()

array(['6:00 am - 9:59 am', '10:00 am - 2:59 pm', '12:00 am - 4:59 am',
       '5:00 am - 5:59 am', '3:00 pm - 6:59 pm', '7:00 pm - 11:59 pm'],
      dtype=object)

In [59]:
clean_string('7:00 pm - 11:59 pm')

'700pm1159pm'

In [60]:
mydata_df['TIME_BOARDED_Clean']=mydata_df['TIME_BOARDED'].apply(clean_string)

In [ ]:
mydata_df['Time_']

In [61]:

time_period_mapping = {
    '600am959am':'AM',
    '1000am259pm':'MID',
    '1200am459am':'OWL',
    '500am559am': 'EARLY AM',
    '300pm659pm': 'PM',
    '700pm1159pm': 'EVENING'
}

In [62]:
time_period_code_mapping = {
    'PM':4,
    'EVENING':5,
    'MID':3,
    'AM': 2,
    'EARLY AM': 1,
    'OWL': 0
}

In [63]:
mydata_df['TIME PERIOD'] = mydata_df['TIME_BOARDED_Clean'].map(time_period_mapping)
mydata_df['TIME PERIOD[Code]'] = mydata_df['TIME PERIOD'].map(time_period_code_mapping)

In [64]:
mydata_df[['TIME PERIOD','TIME PERIOD[Code]']]

,TIME PERIOD,TIME PERIOD[Code]
0,AM,2
1,MID,3
2,MID,3
3,MID,3
4,MID,3
...,...,...
6796,EVENING,5
6797,EARLY AM,1
6798,EARLY AM,1
6799,EARLY AM,1


In [65]:
draft_df=pd.read_excel('o2o_20241022_vta_rail_draft (2).xlsx',sheet_name='O2O_DATA_PWBI')
draft_df.head()

,id,date,final_route_surveyed_code,final_route_surveyed,final_on_stop_id,final_on_stop_lat,final_on_stop_long,final_on_stop_name,final_off_stop_id,final_off_stop_name,...,BOARD_FLG,ALIGHT_FLG,ROUTE_DESCRIPTION[CODE].1,ETC_ROUTE_NAME.1,BOARD_STOP[CODE].1,ALIGHT_STOP[CODE].1,RT_CODE_FLG.1,RT_NAME_FLG.1,BOARD_FLG.1,ALIGHT_FLG.1
0,48577,2024-09-25,VTA_2_BlueLine_00,Blue Line - Baypointe - Santa Teresa [Northbou...,VTA_2_BlueLine_00_4736,37.236668,-121.789141,Santa Teresa Station,VTA_2_BlueLine_00_4743,Tamien Station,...,0,1,VTA_2_BlueLine_00,Blue Line - Baypointe - Santa Teresa [Northbou...,VTA_2_BlueLine_00_4736,VTA_2_BlueLine_00_4743,0,0,0,0
1,48580,2024-09-25,VTA_2_BlueLine_01,Blue Line - Baypointe - Santa Teresa [Southbou...,VTA_2_BlueLine_01_4764,37.394082,-121.933837,Orchard Station,VTA_2_BlueLine_01_4768,Metro/Airport Station,...,0,0,VTA_2_BlueLine_01,Blue Line - Baypointe - Santa Teresa [Southbou...,VTA_2_BlueLine_01_4764,VTA_2_BlueLine_01_4768,0,0,0,0
2,48587,2024-09-25,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,VTA_2_OrangeLine_00_4798,37.403555,-121.974058,Great America Station,VTA_2_OrangeLine_00_4760,Baypointe Station,...,0,0,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,VTA_2_OrangeLine_00_4798,VTA_2_OrangeLine_00_4760,0,0,0,0
3,48588,2024-09-25,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,VTA_2_OrangeLine_00_4802,37.412348,-121.928312,Cisco Station,VTA_2_OrangeLine_00_5242,Alum Rock Station,...,1,1,-,-,-,-,1,1,1,1
4,48589,2024-09-25,VTA_2_OrangeLine_00,Orange Line - Mountain View - Alum Rock [Eastb...,VTA_2_OrangeLine_00_4798,37.403555,-121.974058,Great America Station,VTA_2_OrangeLine_00_5241,Mckee Station,...,0,1,-,-,-,-,1,1,1,1


In [66]:
draft_df.columns

Index(['id', 'date', 'final_route_surveyed_code', 'final_route_surveyed',
       'final_on_stop_id', 'final_on_stop_lat', 'final_on_stop_long',
       'final_on_stop_name', 'final_off_stop_id', 'final_off_stop_name',
       'final_off_stop_lat', 'final_off_stop_long', 'TIME_BOARDED_Code_',
       'TIME_BOARDED', 'ROUTE_DESCRIPTION[CODE]', 'ETC_ROUTE_NAME',
       'BOARD_STOP[CODE]', 'ALIGHT_STOP[CODE]', 'RT_CODE_FLG', 'RT_NAME_FLG',
       'BOARD_FLG', 'ALIGHT_FLG', 'ROUTE_DESCRIPTION[CODE].1',
       'ETC_ROUTE_NAME.1', 'BOARD_STOP[CODE].1', 'ALIGHT_STOP[CODE].1',
       'RT_CODE_FLG.1', 'RT_NAME_FLG.1', 'BOARD_FLG.1', 'ALIGHT_FLG.1'],
      dtype='object')

In [67]:
mydata_df.columns

Index(['id', 'Date', 'Day', 'TIME_BOARDED_Code_', 'TIME_BOARDED',
       'TIME PERIOD', 'TIME PERIOD[Code]', 'ROUTE_DESCRIPTION[CODE]',
       'ETC_ROUTE_NAME', 'Line_Value', 'BOARD_STOP[CODE]', 'Board_Seq',
       'BOARD_STOP_NAME', 'BOARD_STOP_ON_LAT', 'BOARD_STOP_ON_LONG',
       'ALIGHT_STOP[CODE]', 'Alight_Seq', 'ALIGHT_STOP_NAME',
       'ALIGHT_STOP_ON_LAT', 'ALIGHT_STOP_ON_LONG', 'TIME_BOARDED_Clean'],
      dtype='object')

In [68]:
merged_df = pd.merge(
    draft_df[['id', 'TIME_BOARDED_Code_', 'TIME_BOARDED']], 
    mydata_df[['id', 'TIME PERIOD[Code]', 'TIME PERIOD']], on='id'
)

In [69]:
merged_df.columns

Index(['id', 'TIME_BOARDED_Code_', 'TIME_BOARDED', 'TIME PERIOD[Code]',
       'TIME PERIOD'],
      dtype='object')

In [70]:
# merged_df.to_csv('Merged_O2O_Time_Data.csv',index=False)